In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('Tweets.csv')


# Inspect
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df.head())

(14640, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  object 
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object 
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object 
 6   airline_sentiment_gold        40 non-null     object 
 7   name                          14640 non-null  object 
 8   negativereason_gold           32 non-null     object 
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  object 
 11  tweet_coord                   1019 non-null   object 
 12  tweet_created                 14640 non-null  ob

In [6]:
df['airline_sentiment'].value_counts()

,count
airline_sentiment,
negative,9178
neutral,3099
positive,2363


In [7]:
df['label'] = df['airline_sentiment'].map({'negative': 0, 'neutral': 1, 'positive': 2})

In [8]:
import re
df['clean_text'] = df['text'].apply(lambda x: re.sub(r'@\S+', '', x))

print(df[['text', 'clean_text']].head())

                                                text  \
0                @VirginAmerica What @dhepburn said.   
1  @VirginAmerica plus you've added commercials t...   
2  @VirginAmerica I didn't today... Must mean I n...   
3  @VirginAmerica it's really aggressive to blast...   
4  @VirginAmerica and it's a really big bad thing...   

                                          clean_text  
0                                        What  said.  
1   plus you've added commercials to the experien...  
2   I didn't today... Must mean I need to take an...  
3   it's really aggressive to blast obnoxious "en...  
4           and it's a really big bad thing about it  


In [9]:
df['clean_text'] = df['clean_text'].str.lower()

In [10]:
print(df[['text', 'clean_text']].head())


                                                text  \
0                @VirginAmerica What @dhepburn said.   
1  @VirginAmerica plus you've added commercials t...   
2  @VirginAmerica I didn't today... Must mean I n...   
3  @VirginAmerica it's really aggressive to blast...   
4  @VirginAmerica and it's a really big bad thing...   

                                          clean_text  
0                                        what  said.  
1   plus you've added commercials to the experien...  
2   i didn't today... must mean i need to take an...  
3   it's really aggressive to blast obnoxious "en...  
4           and it's a really big bad thing about it  


In [11]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import DistilBertTokenizer

# Step 1: Split ONCE, before any tokenization
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42
)

# Step 2: LSTM tokenization — fit ONLY on training text
keras_tokenizer = Tokenizer(num_words=7000)
keras_tokenizer.fit_on_texts(X_train_text)   # fit only on train

X_train_lstm = keras_tokenizer.texts_to_sequences(X_train_text)
X_train_lstm = pad_sequences(X_train_lstm, maxlen=15)

X_test_lstm = keras_tokenizer.texts_to_sequences(X_test_text)   # transform only, no fitting
X_test_lstm = pad_sequences(X_test_lstm, maxlen=15)

# Step 3: BERT tokenization — no fitting needed, just transforming
bert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

X_train_bert = bert_tokenizer(
    list(X_train_text), padding='max_length', truncation=True, max_length=15, return_tensors='pt'
)
X_test_bert = bert_tokenizer(
    list(X_test_text), padding='max_length', truncation=True, max_length=15, return_tensors='pt'
)

# Sanity checks
print(X_train_lstm.shape, X_test_lstm.shape)
print(X_train_bert['input_ids'].shape, X_test_bert['input_ids'].shape)
print(y_train.shape, y_test.shape)

(11712, 15) (2928, 15)
torch.Size([11712, 15]) torch.Size([2928, 15])
(11712,) (2928,)


In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

model_lstm = Sequential([
    Input(shape=(15,)),
    Embedding(input_dim=7000, output_dim=64),
    LSTM(64),
    Dense(3, activation='softmax')
])

model_lstm.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 15, 64)         │       448,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 481,219 (1.84 MB)

 Trainable params: 481,219 (1.84 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model_lstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_lstm = model_lstm.fit(X_train_lstm, y_train, validation_split=0.2, epochs=7)

Epoch 1/7
293/293 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.7028 - loss: 0.7194 - val_accuracy: 0.7618 - val_loss: 0.5930
Epoch 2/7
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8135 - loss: 0.4658 - val_accuracy: 0.7802 - val_loss: 0.5681
Epoch 3/7
293/293 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8692 - loss: 0.3462 - val_accuracy: 0.7717 - val_loss: 0.5850
Epoch 4/7
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9052 - loss: 0.2671 - val_accuracy: 0.7618 - val_loss: 0.6947
Epoch 5/7
293/293 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9213 - loss: 0.2155 - val_accuracy: 0.7448 - val_loss: 0.7731
Epoch 6/7
293/293 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9395 - loss: 0.1708 - val_accuracy: 0.7571 - val_loss: 0.8271
Epoch 7/7
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9489 - loss: 0.1442 - val_accuracy: 0.7443 - val_loss: 0.9280


In [14]:
!pip install transformers

In [15]:
import tensorflow as tf
print(tf.__version__)

import transformers
print(transformers.__version__)

2.20.0
5.17.0


In [16]:
!pip install transformers[tf] --upgrade

In [17]:
!pip install tf-keras

In [18]:
from transformers import DistilBertForSequenceClassification
print("PyTorch version works fine")

PyTorch version works fine


In [19]:
import torch

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TweetDataset(X_train_bert, y_train)
test_dataset = TweetDataset(X_test_bert, y_test)

In [20]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer

model_bert = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=3
)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy='epoch',
)

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.612540,0.518599
2,0.463446,0.563429
3,0.199928,0.742824


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2196, training_loss=0.3841413713326654, metrics={'train_runtime': 157.943, 'train_samples_per_second': 222.46, 'train_steps_per_second': 13.904, 'total_flos': 136361060254080.0, 'train_loss': 0.3841413713326654, 'epoch': 3.0})

In [21]:
predictions = trainer.predict(test_dataset)
print(predictions.predictions.shape)

(2928, 3)


In [22]:
import numpy as np

predicted_labels_bert = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import accuracy_score, precision_score, recall_score

print("Accuracy:", accuracy_score(y_test, predicted_labels_bert))
print("Precision:", precision_score(y_test, predicted_labels_bert, average='weighted'))
print("Recall:", recall_score(y_test, predicted_labels_bert, average='weighted'))

Accuracy: 0.7991803278688525
Precision: 0.8000200374083614
Recall: 0.7991803278688525
